# Notebook funcional - Orquestador RAG

Este notebook permite ejecutar una consulta end-to-end sobre las colecciones de Qdrant ya indexadas.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

src_path = Path.cwd().resolve().parent  # desde src/notebooks -> src
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

load_dotenv(src_path / ".env")
os.chdir(src_path)

In [2]:
from orchestration.orchestrator import Orchestrator, DEFAULT_TOP_K
from storage.qdrant_store import QdrantStore

qdrant_host = os.getenv('QDRANT_HOST', 'localhost').strip() or 'localhost'
qdrant_port = int(os.getenv('QDRANT_PORT', '6333'))

store = QdrantStore(host=qdrant_host, port=qdrant_port)
orchestrator = Orchestrator(store=store)

print(f'Orquestador listo. Qdrant: {qdrant_host}:{qdrant_port} | TOP_K default: {DEFAULT_TOP_K}')

Orquestador listo. Qdrant: localhost:6333 | TOP_K default: 11


In [4]:
question = 'Que problematicas recurrentes aparecen en las sedes?'

result = orchestrator.answer(question=question, top_k=DEFAULT_TOP_K)

print('Pregunta:')
print(question)
print('\nRespuesta:')
print(result['answer'])

Pregunta:
Que problematicas recurrentes aparecen en las sedes?

Respuesta:
Las principales problemáticas recurrentes son:

- Sucursales desordenadas (reportado en BOG-SUBA-02, MED-ENVIGADO-02, CAR-MANGA-02, CALI-NORTE-02).  
- Personal poco amable.  
- Atención rápida pero poco clara.  
- Congestión en horas pico.  
- Servicio demorado en caja.

Fuentes: reviews


In [5]:
print('Fuentes priorizadas:', result['route']['prioritized_sources'])
print('Fuentes usadas:', result['sources_used'])
print('Chunks enviados al modelo:', len(result['chunks']))

for idx, chunk in enumerate(result['chunks'][:5], start=1):
    preview = chunk['text'][:180].replace('\n', ' ')
    print(f"{idx}. source={chunk['source']} score={chunk['score']:.4f} text={preview}...")

Fuentes priorizadas: ['reviews']
Fuentes usadas: ['reviews']
Chunks enviados al modelo: 11
1. source=reviews score=0.4421 text=Usuario user_29. Sede BOG-SUBA-02. Comentario: La sucursal estaba desordenada....
2. source=reviews score=0.4390 text=Usuario user_16. Sede BOG-SUBA-02. Comentario: La sucursal estaba desordenada....
3. source=reviews score=0.4249 text=Usuario user_27. Sede MED-ENVIGADO-02. Comentario: La sucursal estaba desordenada....
4. source=reviews score=0.4162 text=Usuario user_17. Sede CAR-MANGA-02. Comentario: La sucursal estaba desordenada....
5. source=reviews score=0.4051 text=Usuario user_6. Sede CALI-NORTE-02. Comentario: La sucursal estaba desordenada....
